# ACE F3 — Reproduksi evolusi kandidat DSL (CPU, Rp0)
Notebook ini mereproduksi eksperimen A/B/C pada tugas **development dan validation F2**, bukan held-out F4.
**Batas klaim:** C memodifikasi konfigurasi DSL dua bit yang disusun manusia; ini bukan penemuan algoritma otonom oleh LLM.
Jalankan **Runtime → Run all**. Bila selesai, unduh `ACE_F3_reproduced_evidence.zip` dan unggah ke percakapan. Tidak memerlukan API key, pip, atau koneksi eksternal.


In [ ]:
# ACE F3 - local CPU verification, no API token, no network dependency.
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, tempfile, zipfile

WORK=Path(tempfile.mkdtemp(prefix="ACE_F3_COLAB_"))
BUNDLE="..."
SOURCE_SHA={'ace_f1.py': '4ef8a5c06b75a9df00367ce74226443f1c394b29d253d3f977183f8a9df3a46b', 'ace_f2.py': '69f2244939113ca01c1eea82c48e366f809a957ef6efac6f04af1e82e3caf323', 'ace_f3.py': 'dee41a971e49868376fb12719738aa8bb0f3ed507825eeef33d83664dbca7b82', 'test_f1.py': '5c60253d4364b801b777a7501746c7f8ff27c563616e2816306cae057edf8b57', 'test_f2.py': 'dbafd4c59a2ace6c32d54eb211d79eff3cb3ef555b4db47bd3425a4b8634e93d', 'test_f3.py': 'aece476c214c4a11656360102d3f515797f84c70034d4ff9369d2528ba046fce'}
archive=zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE)))
for item in archive.infolist():
    if item.filename not in SOURCE_SHA or item.is_dir():
        raise ValueError("Unexpected package item")
    data=archive.read(item)
    if hashlib.sha256(data).hexdigest()!=SOURCE_SHA[item.filename]:
        raise ValueError("Embedded source SHA mismatch: "+item.filename)
    (WORK/item.filename).write_bytes(data)

def execute(args, log):
    command=[sys.executable,*args]
    cp=subprocess.run(command, cwd=WORK, capture_output=True, text=True, timeout=300)
    (WORK/log).write_text(cp.stdout+"\n"+cp.stderr,encoding="utf8")
    if cp.returncode!=0:
        print(cp.stdout[-5000:]);print(cp.stderr[-5000:])
        raise RuntimeError("Command failed: "+str(command))
    return cp

execute(["-m","unittest","-v","test_f1","test_f2","test_f3"],"test_log.txt")
execute(["ace_f3.py","all","--root","evidence"],"run_log.txt")
execute(["ace_f3.py","verify","--root","evidence"],"verify_log.txt")
summary=json.loads((WORK/"evidence/reports/f3_metrics.json").read_text())
for mode in ("A_CONTROL","B_ADAPT","C_EVOLVE"):
    correct=sum(v["correct"] for v in summary["scores"][mode].values())
    total=sum(v["total"] for v in summary["scores"][mode].values())
    print(mode,correct,"/",total)
print("F4 held-out:",summary["heldout_f4"])
print("46 acceptance tests + 9 run replays completed")

# Signed manifests are checksums (tamper evidence), not digital signatures.
files={p.relative_to(WORK).as_posix():p for p in sorted(WORK.rglob("*")) if p.is_file() and "__pycache__" not in p.parts}
manifest={name:hashlib.sha256(path.read_bytes()).hexdigest() for name,path in files.items()}
(WORK/"SHA256SUMS.json").write_text(json.dumps(manifest,indent=2,sort_keys=True)+"\n")
output=WORK/"ACE_F3_reproduced_evidence.zip"
with zipfile.ZipFile(output,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for name,path in files.items():z.write(path,arcname=name)
    z.write(WORK/"SHA256SUMS.json",arcname="SHA256SUMS.json")
with zipfile.ZipFile(output) as z:
    for name,sha in manifest.items():
        if hashlib.sha256(z.read(name)).hexdigest()!=sha:raise AssertionError("ZIP verify failed")
print("EVIDENCE ZIP:",output)
try:
    from google.colab import files as colab_files
    colab_files.download(str(output))
except ImportError:
    print("Not in Colab; ZIP remains at",output)
